# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [35]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [36]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [37]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [60]:
link_system_prompt = """
You are provided with a list of blog/article links found on a webpage.
You are able to decide which of the links are relevant blog posts or articles or news,
and provide a brief summary of what each writeup is likely about based on its URL.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "blog post", "url": "https://full.url/blog/my-post", "summary": "Brief summary of what this post is likely about"},
        {"type": "article", "url": "https://full.url/article/my-article", "summary": "Brief summary of what this article is likely about"}
    ]
}
"""

In [61]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of blog, article, news links found on the website {url} -
Please review these links and provide a short summary for each relevant blog post or article or news.
Ignore any which is not either of blog or article or news links and also ignore Terms of Service, Privacy, or email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)

    if not links:
        user_prompt += "No blog or article or news links found."
    else:
        user_prompt += "\n".join(links)

    return user_prompt

In [40]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of blog and article links found on the website https://edwarddonner.com -
Please review these links and provide a short summary for each relevant blog post or article.
Ignore any non-blog links, Terms of Service, Privacy, or email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com

In [41]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [42]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
   'summary': "Explores the evolution from an 'AI coder' or 'vibe coder' to an 'agentic engineer', outlining what it means to build autonomous AI agents and the skills, tools, and capabilities involved."},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
   'summary': 'Describes building AI agents using the n8n workflow automation tool and how to create voice-enabled agents, with practical guidance and examples.'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
   'summary': 'Discusses deploying generative AI and agentic AI in production at scale on AWS, covering architecture, reliability, and best practices.'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become

In [43]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [44]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 4 relevant links


{'links': [{'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
   'summary': 'Explores how the coding role is evolving in the AI era, moving from traditional coding toward building agentic AI systems and autonomous agents, with insights on the skills and workflows this shift requires.'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
   'summary': 'Practical guide to using the n8n automation platform to build AI agents and voice agents, including step-by-step approaches to orchestrate agents within workflows.'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
   'summary': 'Discussion of deploying generative AI and agentic AI in production on AWS at scale, covering architecture patterns, performance, security, and operational considerations.'},
  {'type': 'blog post',
  

In [45]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano


KeyboardInterrupt: 

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [46]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://edwarddonner.com"))

In [62]:
blog_summary_system_prompt = """
You are an assistant that analyzes the contents of several blog posts, articles and news from a website
and creates a concise, well-structured summary of each post for readers.
Respond in markdown without code blocks.
For each post include:
- The main topic or thesis
- Key points or takeaways
- Any relevant insights, data, or examples mentioned
"""
# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [64]:
def get_blog_summary_user_prompt(website_name, url):
    user_prompt = f"""
You are looking at a website called: {website_name}
Here are the contents of its blog posts, articles or news;
use this information to create a concise summary of each post in markdown without code blocks.\n\n
"""
    user_prompt += "\n".join(fetch_page_and_all_relevant_links(url))
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [65]:
get_blog_summary_user_prompt("edwarddonner", "https://thinkingmachines.ai/")

Selecting relevant links for https://thinkingmachines.ai/ by calling gpt-5-nano
Found 1 relevant links


'\nYou are looking at a website called: edwarddonner\nHere are the contents of its blog posts, articles or news;\nuse this information to create a concise summary of each post in markdown without code blocks.\n\n\n#\n#\n \nL\na\nn\nd\ni\nn\ng\n \nP\na\ng\ne\n:\n\n\n\n\nT\nh\ni\nn\nk\ni\nn\ng\n \nM\na\nc\nh\ni\nn\ne\ns\n \nL\na\nb\n\n\n\n\nT\nH\nI\nN\nK\nI\nN\nG\n \nM\nA\nC\nH\nI\nN\nE\nS\n\n\nT\ni\nn\nk\ne\nr\n\n\nC\no\nn\nn\ne\nc\nt\ni\no\nn\ni\ns\nm\n\n\nN\ne\nw\ns\n\n\nJ\no\ni\nn\n \nu\ns\n\n\nH\no\nm\ne\n\n\nT\ni\nn\nk\ne\nr\n\n\nC\no\nn\nn\ne\nc\nt\ni\no\nn\ni\ns\nm\n\n\nN\ne\nw\ns\n\n\nJ\no\ni\nn\n \nu\ns\n\n\nT\nh\ni\nn\nk\ni\nn\ng\n\n\nM\na\nc\nh\ni\nn\ne\ns\n\n\nN\nE\nW\n\n\nI\nn\nt\ne\nr\na\nc\nt\ni\no\nn\n \nM\no\nd\ne\nl\ns\n:\n \nA\n \nS\nc\na\nl\na\nb\nl\ne\n \nA\np\np\nr\no\na\nc\nh\n \nt\no\n \nH\nu\nm\na\nn\n-\nA\nI\n \nC\no\nl\nl\na\nb\no\nr\na\nt\ni\no\nn\n\n\nT\nh\ni\nn\nk\ni\nn\ng\n \nM\na\nc\nh\ni\nn\ne\ns\n \nL\na\nb\n \ni\ns\n \na\nn\n \na\nr\nt\ni\nf\ni\nc\ni\n

In [54]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": blog_summary_system_prompt},
            {"role": "user", "content": get_blog_summary_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [67]:
create_brochure("TigerBeetle", "https://tigerbeetle.com/")

Selecting relevant links for https://tigerbeetle.com/ by calling gpt-5-nano
Found 10 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


## Summary of TigerBeetle Website Content and Posts

---

### 1. TigerBeetle Overview and Performance Highlights

**Main Topic:**  
TigerBeetle is a high-performance, distributed OLTP (Online Transaction Processing) database designed specifically for financial transactions.

**Key Points:**  
- Focuses on delivering high throughput, predictable low latency, and cost-efficiency at scale.  
- Achieves up to 1000x faster OLTP performance compared to traditional databases.  
- Designed with first-principles and relentless optimization to move more transactions faster using less hardware.  
- Supports 100K to 500K TPS (transactions per second) with 100ms P100 latency.  
- Provides strict consistency, multi-cloud high availability, and indestructible durability features.  
- Production-ready on Linux, and easily integrates with Python, Java, Node.js, .Net, and Go.  
- Apache 2.0 open-source license without Contributor License Agreements (CLAs).  
- Weekly releases with the latest version noted as 0.17.3 (as of May 4, 2026).

**Insights:**  
- The emphasis on financial transactions underscores its reliability and strict consistency.  
- The ability to run on multiple OS (Linux, macOS, Windows) and support curl-based installation highlights accessibility for developers.  

---

### 2. TigerBeetle Version 0.16.11 Testing and Stability Report

**Main Topic:**  
Analysis and testing report of TigerBeetle version 0.16.11 focusing on stability under different scenarios.

**Key Points:**  
- TigerBeetle 0.16.11 is a distributed OLTP database geared toward financial transactions.  
- Testing was conducted through subsequent versions up to 0.16.30.  
- Identification of seven client and server crashes during testing including a client segfault.  
  
**Insights:**  
- Active testing phase ensures robustness, although some crashes have been found indicating the need for ongoing debugging and development.  
- Transparency about crashes reflects TigerBeetle's commitment to reliability and continuous improvement.

---

### 3. General TigerBeetle Features and Install Instructions

**Main Topic:**  
TigerBeetle’s core features and instructions for installation.

**Key Points:**  
- TigerBeetle emphasizes performance, consistency, availability, durability, and efficiency.  
- Supports multi-cloud deployment for high availability.  
- Installation instructions are straightforward using curl to download and unzip on Linux/macOS/Windows.  
- Seamless integration with major programming languages and environments.  
- No CLA requirement encourages community contributions and adoption.

**Insights:**  
- TigerBeetle is designed for practical deployment scenarios with easy installation and broad language support.  
- The focus on being production-ready and open-source under a permissive license encourages adoption in financial and transactional systems.

---

### Additional Notes

- TigerBeetle positions itself as a revolutionary database solution to power the next 30 years of online transaction processing.  
- The website promotes upcoming events like Systems Distributed ’26 in Boston, indicating an active community and outreach.

---

This summary condenses the main themes and findings from the provided TigerBeetle content, emphasizing its technical strengths, features, and development status.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": blog_summary_system_prompt},
            {"role": "user", "content": get_blog_summary_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("edwarddonner", "https://edwarddonner.com")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links


# Hugging Face Brochure

---

## Who We Are

**Hugging Face** is the AI community building the future of machine learning. We are the collaboration platform for the global machine learning (ML) community — a central hub where engineers, scientists, and AI enthusiasts create, share, discover, and experiment with open-source ML models, datasets, and applications.

Our mission is to empower the next generation of ML engineers, scientists, and end users to learn, collaborate, and innovate in building an open and ethical AI future. With a fast-growing community and some of the most popular open-source ML tools and libraries, Hugging Face is at the heart of the AI revolution.

---

## Our Platform

- **Models:** Browse over 2 million open-source ML models spanning text, image, video, audio, and even 3D modalities. Whether you want to leverage existing models or share your own creations, Hugging Face Hub is your go-to place.

- **Datasets:** Access a diverse collection of over 500,000 datasets for training and benchmarking ML models.

- **Spaces:** Host and run ML applications and demos in a collaborative environment, making it easy to share and showcase your work with the community.

- **Buckets:** New collaborative data storage, designed to support ML workflows directly.

- **Applications & Agents:** Build and explore cutting-edge AI applications including voice cloning supporting 600+ languages and more.

Our platform supports unlimited public hosting and sharing, enabling the community to move faster and build their portfolios to showcase their expertise.

---

## Company Culture

Hugging Face fosters an open, ethical, and collaborative culture. We strongly support community engagement, continuous learning, and innovation. Our team is energized by:

- Pioneering research in state-of-the-art AI and machine learning technologies.
- Building inclusive tools that help democratize AI access and education.
- Encouraging transparency and openness in AI development.

We celebrate diversity of thought and background, committed to empowering individuals across the globe to contribute to and benefit from AI advancements.

---

## Our Customers and Community

Our community includes:

- Individual ML engineers, data scientists, and AI researchers.
- Academic institutions leveraging open datasets and models for research.
- Enterprises scaling AI models and deploying them at production level using our Enterprise & Team plans.
- Startups and developers building AI-powered applications and services.
- Educators and learners using our platform as a learning and teaching tool.

This vibrant and diverse user base actively contributes to our hub, driving innovation, and sharing insights to build a stronger AI ecosystem.

---

## Careers at Hugging Face

Join a cutting-edge AI company with opportunities to work with talented researchers and engineers pushing the limits of machine learning.

- We look for passionate, creative, and collaboration-driven individuals.
- Open roles span software engineering, research science, product management, and more.
- Work remotely or in our hubs, contributing to some of the industry’s most impactful ML products.
- Become part of the AI revolution while working in a supportive, inclusive, and mission-driven environment.

Visit our **Careers page** on the Hugging Face website to explore current openings and learn how you can contribute to shaping the future of AI.

---

## Enterprise Solutions

Hugging Face offers scalable Team & Enterprise Plans designed for organizations seeking to harness AI power securely and efficiently:

- Collaborative private repositories for proprietary models and datasets.
- Enhanced support and SLAs.
- Tools to deploy, manage, and scale ML models and applications in production.
- Integration with existing workflows for faster innovation cycles.

Enterprises across industries rely on Hugging Face for accelerating their AI initiatives while maintaining compliance and control.

---

## Brand & Visual Identity

- **Primary Colors:** Hugging Face’s brand colors include bright yellow (#FFD21E), vibrant orange (#FF9D00), and balanced gray (#6B7280).
- **Logo:** Clean, friendly logos and icons are available in SVG, PNG, and AI formats.
- These assets reflect our warm, innovative, and approachable personality.

---

## Connect with Us

- **Website:** https://huggingface.co  
- **GitHub:** https://github.com/huggingface  
- **Twitter:** https://twitter.com/huggingface  
- **LinkedIn:** https://linkedin.com/company/huggingface  
- **Discord:** Hugging Face community server for real-time chat and support  

---

**Hugging Face – The AI community building the future. Join us today and be part of the machine learning revolution!**

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>